# A research walkthrough

The same thing `tradelab run` does, one step at a time, with the reasoning visible.

The data here is **synthetic**. It was generated with a known drift, volatility and
return autocorrelation, so a strategy performing well on it has demonstrated that the
code works and nothing about markets. Point the loader at your own data to learn
anything else; `data/README.md` describes the format.


## 1. Load, and check before trusting

The validator is the first thing that runs. A backtest on bad data is worse than no
backtest, because it produces a number that looks like a result.


In [ ]:
from pathlib import Path

from tradelab.data.loaders import load_directory

data, reports = load_directory(Path("..") / "data" / "sample")
for report in reports:
    print(report)

In [ ]:
closes = data.field("close")
closes.tail()

## 2. Cut the data before looking at it

The development period is where you are allowed to iterate. The out-of-sample period
is not, and every time you look at it you spend from the same budget.


In [ ]:
from tradelab.data.splits import date_split

development, out_of_sample = date_split(data.index, "2019-12-31")
print(development)
print(out_of_sample)

## 3. What a strategy is allowed to see

`MarketView` is the look-ahead barrier. It is constructed for one bar and has no
accessor that returns anything later than that bar.


In [ ]:
view = data.view(1000)
print(view)
print("last visible bar:", view.now.date())
print("history length:  ", len(view.series("SYN_TREND")))

# There is no way to ask this object for tomorrow. Orders it leads to are
# executed at the *next* bar's open, so even today's close is not tradable.
view.series("SYN_TREND", lookback=5)

## 4. Run one strategy

Costs are on by default and are deliberately not the cheapest available. A backtest
that only works at zero cost is a backtest that does not work.


In [ ]:
from tradelab.backtesting.engine import BacktestConfig, run_backtest
from tradelab.risk.limits import RiskLimits, RiskManager
from tradelab.risk.sizing import VolatilityTarget
from tradelab.strategies import MovingAverageCrossover

development_data = data.slice(development.start, development.end)

strategy = MovingAverageCrossover(fast=50, slow=200, confirmation_bars=2)
risk = RiskManager(
    limits=RiskLimits(max_position_weight=0.35, allow_shorts=False),
    sizer=VolatilityTarget(annual_target=0.10, lookback=60),
)

result = run_backtest(development_data, strategy, risk=risk, config=BacktestConfig())
result.equity_curve.tail()

## 5. Read the result honestly

The tear sheet prints the assumptions underneath the numbers, because a performance
table with no cost model beside it is not something anyone can check.


In [ ]:
from tradelab.analytics.tearsheet import report

print(report(result, title="MA crossover, development period"))

### The number under the number

A Sharpe ratio is an estimate. `sharpe_standard_error` says how rough an estimate,
and `probabilistic_sharpe_ratio` turns it into the question actually being asked:
given this sample, how likely is the true Sharpe to be above zero?


In [ ]:
from tradelab.analytics.metrics import summarise

summary = summarise(result)
print(f"Sharpe        {summary.sharpe_ratio:>7.2f}")
print(f"standard error{summary.sharpe_standard_error:>7.2f}")
print(f"P(Sharpe > 0) {summary.probabilistic_sharpe_ratio:>7.1%}")
print()
print("A Sharpe of 0.5 with a standard error of 0.25 is a 95% interval of roughly")
print("0.0 to 1.0. Two decimal places is more precision than the sample supports.")

## 6. Compare against doing nothing clever

Every result should be reported next to buy-and-hold on the same data over the same
period through the same cost model. A strategy returning 7% a year is not interesting
until you know what holding the universe returned.


In [ ]:
from tradelab.analytics.tearsheet import comparison_table, to_markdown
from tradelab.strategies import BuyAndHold

benchmark = run_backtest(
    development_data, BuyAndHold(), config=BacktestConfig(rebalance_tolerance=0.10)
)

table = comparison_table({"ma_crossover": summary, "buy_and_hold": summarise(benchmark)})
table.index.name = "strategy"
print(to_markdown(table))

## 7. Does it survive worse assumptions?

Slippage is an assumption, not a measurement. The useful question is not what the
base case says but whether the result exists at four times the cost.


In [ ]:
from tradelab.backtesting.costs import BpsCommission, NoCommission
from tradelab.backtesting.engine import with_costs
from tradelab.backtesting.slippage import FixedBpsSlippage, NoSlippage

base = BacktestConfig()
scenarios = {
    "zero cost": with_costs(base, NoCommission(), NoSlippage()),
    "base": base,
    "pessimistic": with_costs(base, BpsCommission(8.0), FixedBpsSlippage(25.0)),
}

for label, config in scenarios.items():
    run = run_backtest(development_data, strategy, risk=risk, config=config)
    print(f"{label:<12} CAGR {summarise(run).cagr:>7.2%}")

## 8. Only now, the out-of-sample period

Once. The run starts before the boundary so the indicators are warm on the first
measured bar; the records are trimmed to the measurement window afterwards.


In [ ]:
from tradelab.backtesting.engine import run_backtest
from tradelab.config import DataConfig, RunConfig
from tradelab.research import trim_to_split

full = run_backtest(data, strategy, risk=risk, config=BacktestConfig())
config = RunConfig(
    name="walkthrough",
    strategy=strategy,
    data=DataConfig(directory="data/sample", development_end="2019-12-31"),
    backtest=BacktestConfig(),
    risk=risk,
)
oos = summarise(trim_to_split(full, config, "out_of_sample"))

print(f"development   CAGR {summary.cagr:>7.2%}  Sharpe {summary.sharpe_ratio:>5.2f}")
print(f"out of sample CAGR {oos.cagr:>7.2%}  Sharpe {oos.sharpe_ratio:>5.2f}")

### What to conclude

Usually: less than you hoped. A large gap between the two rows is the normal outcome
and is the reason the split exists. If out-of-sample is *better*, that is a coincidence
of one boundary date rather than good news - use `walk_forward` to score the rule over
many boundaries before believing either number.

And none of this is evidence about markets. The data is synthetic.


## 9. Charts


In [ ]:
from tradelab.analytics import plots

figure = plots.equity_curves(
    {
        "ma_crossover": result.equity_curve["equity"],
        "buy_and_hold": benchmark.equity_curve["equity"],
    },
    title="Development period (synthetic data)",
)
figure

In [ ]:
plots.exposure_chart(result.equity_curve)

## 10. Adding a strategy

Two methods. `warmup` says how many bars of history the indicators need before a
signal means anything, and the engine holds the book flat until then. `generate`
returns target weights as a signed fraction of equity.

Nothing else in the framework needs to change, and the look-ahead tests in
`tests/test_no_lookahead.py` will apply to it as soon as it is added to their list.


In [ ]:
from tradelab.data.market import MarketView
from tradelab.strategies.base import Strategy


class AboveItsOwnAverage(Strategy):
    """Equal weight across whatever is trading above its 100-bar mean."""

    name = "above_average_demo"

    def __init__(self, window: int = 100) -> None:
        self.window = window

    @property
    def warmup(self) -> int:
        return self.window + 1

    def generate(self, view: MarketView) -> dict[str, float]:
        picks = []
        for symbol in view.tradable():
            closes = view.series(symbol, lookback=self.window)
            if len(closes) == self.window and closes.iloc[-1] > closes.mean():
                picks.append(symbol)
        return {symbol: 1.0 / len(picks) for symbol in picks} if picks else {}


demo = run_backtest(development_data, AboveItsOwnAverage(), config=BacktestConfig())
print(report(demo, title="a strategy written in twenty lines"))